In [1]:
pip install --user tqdm

In [2]:
# conda install -c conda-forge tqdm

ValueError: The python kernel does not appear to be a conda environment.  Please use ``%pip install`` instead.

In [3]:
# 코드 6-1 필요한 라이브러리 호출

import torch
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.autograd import Variable
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
import os
import cv2
from PIL import Image
from tqdm import tqdm_notebook as tqdm
import random
from matplotlib import pyplot as plt

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
# 코드 6-2 이미지 데이터 전처리

class ImageTransform():
    def __init__(self, resize, mean, std):
        self.data_transform = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(resize, scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ]),
            'val': transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(resize),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        }

    def __call__(self, img, phase):
        return self.data_transform[phase](img)

In [ ]:
# 코드 6-3 이미지 데이터셋을 불러온 후 훈련, 검증, 테스트로 분리

cat_directory = r'../chap06/data/dogs-vs-cats/Cat/'
dog_directory = r'../chap06/data/dogs-vs-cats/Dog/'

cat_images_filepaths = sorted([os.path.join(cat_directory, f) for f in
                               os.listdir(cat_directory)])
dog_images_filepaths = sorted([os.path.join(dog_directory, f) for f in
                               os.listdir(dog_directory)])
images_filepaths = [*cat_images_filepaths, *dog_images_filepaths]
correct_images_filepaths = [i for i in images_filepaths if cv2.imread(i) is not None]

random.seed(42)
random.shuffle(correct_images_filepaths)
train_images_filepaths = correct_images_filepaths[:400]
val_images_filepaths = correct_images_filepaths[400:-10]
test_images_filepaths = correct_images_filepaths[-10:]
print(len(train_images_filepaths), len(val_images_filepaths), len(test_images_filepaths))

In [7]:
# 예시
import os
list_path = ['c:\\', 'Temp', 'user']
folder_path=os.path.join(*list_path)
folder_path

'c:\\/Temp/user'

In [6]:
# 예시
import numpy as np

np.random.seed(101)
np.random.randint(low=1, high=10, size=10)

array([2, 7, 8, 9, 5, 9, 6, 1, 6, 9])

In [8]:
# 예시
np.random.seed(100)
np.random.randint(low=1, high=10, size=10)

array([9, 9, 4, 8, 8, 1, 5, 3, 6, 3])

In [9]:
# 코드 6-4 테스트 데이터셋 이미지 확인 함수

def display_image_grid(images_filepaths, predicted_labels=(), cols=5):
    rows = len(images_filepaths) // cols
    figure, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 6))
    for i, image_filepath in enumerate(images_filepaths):
        image = cv2.imread(image_filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        true_label = os.path.normpath(image_filepath).split(os.sep)[-2]
        predicted_label = predicted_labels[i] if predicted_labels else true_label
        color = "green" if true_label == predicted_label else "red"
        ax.ravel()[i].imshow(image)
        ax.ravel()[i].set_title(predicted_label, color=color)
        ax.ravel()[i].set_axis_off()
    plt.tight_layout()
    plt.show()

In [10]:
# 코드 6-5 테스트 데이터셋 이미지를 출력

display_image_grid(test_images_filepaths)

NameError: name 'test_images_filepaths' is not defined

In [11]:
# 코드 6-6 이미지 데이터셋 클래스 정의

class DogvsCatDataset(Dataset):
    def __init__(self, file_list, transform=None, phase='train'):
        self.file_list = file_list
        self.transform = transform
        self.phase = phase

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path)
        img_transformed = self.transform(img, self.phase)
        label = img_path.split('/')[-1].split('.')[0]
        if label == 'dog':
            label = 1
        elif label == 'cat':
            label = 0
        return img_transformed, label

In [ ]:
# 코드 6-7 변수 값 정의

size = 224
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
batch_size = 32

In [ ]:
# 코드 6-8 이미지 데이터셋 정의

train_dataset = DogvsCatDataset(train_images_filepaths, transform=ImageTransform(size,
                                      mean, std), phase='train')
val_dataset = DogvsCatDataset(val_images_filepaths, transform=ImageTransform(size,
                                      mean, std), phase='val')

index = 0
print(train_dataset.__getitem__(index)[0].size())
print(train_dataset.__getitem__(index)[1])

In [ ]:
# 코드 6-9 데이터로더 정의

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
dataloader_dict = {'train': train_dataloader, 'val': val_dataloader}

batch_iterator = iter(train_dataloader)
inputs, label = next(batch_iterator)
print(inputs.size())
print(label)

In [ ]:
# 코드 6-10 모델의 네트워크 클래스

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()

        self.cnn1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1,
                              padding=0)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)
        self.cnn2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5,
                              stride=1, padding=0)
        self.relu2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)

        self.fc1 = nn.Linear(32*53*53, 512)
        self.relu5 = nn.ReLU()
        self.fc2 = nn.Linear(512, 2)
        self.output = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.cnn1(x)
        out = self.relu1(out)
        out = self.maxpool1(out)
        out = self.cnn2(out)
        out = self.relu2(out)
        out = self.maxpool2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.output(out)
        return out

In [ ]:
# p.259 | 코드 6-11 | LeNet 모델 객체 생성 및 모델 구조 출력

model = LeNet()
print(model)

In [ ]:
# p.260 | 예제 | torchsummary 라이브러리 설치

pip install torchsummary

In [ ]:
# p.260 | 코드 6-12 | torchsummary를 이용한 모델의 네트워크 구조 확인

from torchsummary import summary
summary(model, input_size=(3,224,224))

In [ ]:
# p.261 | 코드 6-13 | 모델에서 학습 가능한 파라미터 수 확인

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

In [ ]:
# p.262 | 코드 6-14 | SGD 옵티마이저와 CrossEntropyLoss 손실 함수 정의

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
criterion = nn.CrossEntropyLoss()

In [ ]:
# p.262 | 코드 6-15 | 모델의 파라미터와 손실 함수를 CPU에 할당

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
# p.262~263 | 코드 6-16 | 훈련과 검증을 반복하는 모델 학습 함수 정의

def train_model(model, dataloader_dict, criterion, optimizer, num_epoch):
    since = time.time()
    best_acc = 0.0

    for epoch in range(num_epoch):
        print('Epoch {}/{}'.format(epoch+1, num_epoch))
        print('-'*20)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            epoch_loss = 0.0
            epoch_corrects = 0

            for inputs, labels in tqdm(dataloader_dict[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                epoch_loss += loss.item() * inputs.size(0)
                epoch_corrects += torch.sum(preds == labels.data)

            epoch_loss = epoch_loss / len(dataloader_dict[phase].dataset)
            epoch_acc = epoch_corrects.double() / len(dataloader_dict[phase].dataset)

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(phase, epoch_loss, epoch_acc))

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f}'.format(best_acc))
    return model

In [ ]:
# p.264 | CrossEntropyLoss의 reduction을 mean으로 지정

criterion = nn.CrossEntropyLoss(reduction='mean')

In [12]:
# p.264 | 코드 6-17 | epoch을 10으로 설정하고 train_model 함수를 호출하여 모델 학습

import time

num_epoch = 10
model = train_model(model, dataloader_dict, criterion, optimizer, num_epoch)

NameError: name 'train_model' is not defined

In [ ]:
# p.265~266| 코드 6-18 | 테스트 데이터에 모델을 적용하고 예측 결과를 CSV 파일로 저장

import pandas as pd

id_list = []
pred_list = []
_id = 0

with torch.no_grad():
    for test_path in tqdm(test_images_filepaths):
        img = Image.open(test_path)
        _id = test_path.split('/')[-1].split('.')[1]
        transform = ImageTransform(size, mean, std)
        img = transform(img, phase='val')
        img = img.unsqueeze(0)
        img = img.to(device)

        model.eval()
        outputs = model(img)
        preds = F.softmax(outputs, dim=1)[:, 1].tolist()
        id_list.append(_id)
        pred_list.append(preds[0])

res = pd.DataFrame({
    'id': id_list,
    'label': pred_list
})

res.sort_values(by='id', inplace=True)
res.reset_index(drop=True, inplace=True)

res.to_csv('../chap06/data/LeNet', index=False)

In [ ]:
# p.266 | softmax를 적용하고 두 번째 클래스의 예측값을 리스트로 변환

F.softmax(outputs, dim=1)[:, 1].tolist()

In [ ]:
# p.267 | 코드 6-19 | DataFrame에 저장된 테스트 데이터셋 예측 결과 중 앞의 10개 확인

res.head(10)

In [ ]:
# p.267~268 | 코드 6-20 | 테스트 데이터셋의 예측 결과를 이미지로 출력하는 함수 정의

class_ = classes = {0:'cat', 1:'dog'}

def display_image_grid(images_filepaths, predicted_labels=(), cols=5):
    rows = len(images_filepaths) // cols
    figure, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 6))
    for i, image_filepath in enumerate(images_filepaths):
        image = cv2.imread(image_filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        a = random.choice(res['id'].values)
        label = res.loc[res['id'] == a, 'label'].values[0]
        if label > 0.5:
            label = 1
        else:
            label = 0

        ax.ravel()[i].imshow(image)
        ax.ravel()[i].set_title(class_[label])
        ax.ravel()[i].set_axis_off()

    plt.tight_layout()
    plt.show()

In [ ]:
# p.268 | 코드 6-21 | 테스트 데이터셋에 대한 LeNet 예측 결과를 이미지로 출력

display_image_grid(test_images_filepaths)

In [ ]:
# p.271 | 코드 6-22 | AlexNet 실습에 필요한 라이브러리 호출 및 device 설정

import torch
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.autograd import Variable
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
import os
import cv2
import random
from PIL import Image
from tqdm import tqdm_notebook as tqdm

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# p.272 | 코드 6-23 | AlexNet 학습 및 검증 이미지에 적용할 데이터 전처리 정의

class ImageTransform():
    def __init__(self, resize, mean, std):
        self.data_transform = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(resize, scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ]),
            'val': transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(resize),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        }

    def __call__(self, img, phase):
        return self.data_transform[phase](img)

In [ ]:
# p.273 | 코드 6-24 | 이미지 경로를 불러와 훈련, 검증, 테스트 데이터로 분리

cat_directory = '../chap06/data/dogs-vs-cats/Cat/'
dog_directory = '../chap06/data/dogs-vs-cats/Dog/'

cat_images_filepaths = sorted([os.path.join(cat_directory, f) for f in os.listdir(cat_directory)])
dog_images_filepaths = sorted([os.path.join(dog_directory, f) for f in os.listdir(dog_directory)])

images_filepaths = [*cat_images_filepaths, *dog_images_filepaths]
correct_images_filepaths = [i for i in images_filepaths if cv2.imread(i) is not None]

random.seed(42)
random.shuffle(correct_images_filepaths)
train_images_filepaths = correct_images_filepaths[:400]
val_images_filepaths = correct_images_filepaths[400:-10]
test_images_filepaths = correct_images_filepaths[-10:]

print(len(train_images_filepaths), len(val_images_filepaths), len(test_images_filepaths))

In [ ]:
# p.274 | 코드 6-25 | Dataset을 상속하여 Cat/Dog 이미지용 커스텀 데이터셋 클래스 정의

class DogvsCatDataset(Dataset):
    def __init__(self, file_list, transform=None, phase='train'):
        self.file_list = file_list
        self.transform = transform
        self.phase = phase

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path)
        img_transformed = self.transform(img, self.phase)

        label = img_path.split('/')[-1].split('.')[0]
        if label == 'dog':
            label = 1
        elif label == 'cat':
            label = 0

        return img_transformed, label

In [ ]:
# p.274 | 코드 6-26 | 이미지 전처리에 사용할 변수 값 정의

size = 256
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
batch_size = 32

In [ ]:
# p.274~275 | 코드 6-27 | 훈련, 검증, 테스트 데이터셋 정의

train_dataset = DogvsCatDataset(train_images_filepaths, transform=ImageTransform(size,
                                      mean, std), phase='train')

val_dataset = DogvsCatDataset(val_images_filepaths, transform=ImageTransform(size,
                                      mean, std), phase='val')

test_dataset = DogvsCatDataset(val_images_filepaths, transform=ImageTransform(size,
                                      mean, std), phase='val')

index = 0
print(train_dataset.__getitem__(index)[0].size())
print(train_dataset.__getitem__(index)[1])

In [ ]:
# p.275 | 코드 6-28 | DataLoader를 이용해 데이터셋을 메모리로 불러옴

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

dataloader_dict = {'train': train_dataloader, 'val': val_dataloader}

batch_iterator = iter(train_dataloader)
inputs, label = next(batch_iterator)

print(inputs.size())
print(label)

In [ ]:
# p.276 | 코드 6-29 | AlexNet 모델의 네트워크 정의

class AlexNet(nn.Module):
    def __init__(self) -> None:
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))

        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256*6*6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
# p.277 | 코드 6-30 | AlexNet 모델 객체 생성 후 device에 할당

model = AlexNet()
model.to(device)

In [ ]:
# p.278 | 코드 6-31 | 옵티마이저 및 손실 함수 정의

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
criterion = nn.CrossEntropyLoss()

In [ ]:
# p.278 | 코드 6-32 | torchsummary를 이용하여 모델 네트워크 구조 확인

from torchsummary import summary
summary(model, input_size=(3, 256, 256))

In [ ]:
# p.279~280 | 코드 6-33 | 훈련 및 검증을 수행하는 모델 학습 함수 정의

def train_model(model, dataloader_dict, criterion, optimizer, num_epoch):

    since = time.time()
    best_acc = 0.0

    for epoch in range(num_epoch):
        print('Epoch {}/{}'.format(epoch+1, num_epoch))
        print('-'*20)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            epoch_loss = 0.0
            epoch_corrects = 0

            for inputs, labels in tqdm(dataloader_dict[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                epoch_loss += loss.item() * inputs.size(0)
                epoch_corrects += torch.sum(preds == labels.data)

            epoch_loss = epoch_loss / len(dataloader_dict[phase].dataset)
            epoch_acc = epoch_corrects.double() / len(dataloader_dict[phase].dataset)

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(phase, epoch_loss, epoch_acc))

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))

    return model

In [ ]:
# p.281 | 코드 6-34 | epoch을 10으로 설정하여 AlexNet 모델 학습

num_epoch = 10
model = train_model(model, dataloader_dict, criterion, optimizer, num_epoch)

In [ ]:
# p.281~282 | 코드 6-35 | 학습한 AlexNet 모델을 이용하여 테스트 데이터 예측

import pandas as pd

id_list = []
pred_list = []
_id = 0

with torch.no_grad():
    for test_path in tqdm(test_images_filepaths):
        img = Image.open(test_path)
        _id = test_path.split('/')[-1].split('.')[1]
        transform = ImageTransform(size, mean, std)

        img = transform(img, phase='val')
        img = img.unsqueeze(0)
        img = img.to(device)

        model.eval()
        outputs = model(img)
        preds = F.softmax(outputs, dim=1)[:, 1].tolist()

        id_list.append(_id)
        pred_list.append(preds[0])

res = pd.DataFrame({
    'id': id_list,
    'label': pred_list
})

res.to_csv('../chap06/data/alexnet.csv', index=False)

In [ ]:
# p.282 | 코드 6-36 | 테스트 데이터 예측 결과 DataFrame 확인

res.head(10)

In [ ]:
# p.283 | 코드 6-37 | AlexNet의 예측 결과를 이미지로 표현하기 위한 함수 정의

class_ = classes = {0:'cat', 1:'dog'}

def display_image_grid(images_filepaths, predicted_labels=(), cols=5):
    rows = len(images_filepaths) // cols
    figure, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 6))

    for i, image_filepath in enumerate(images_filepaths):
        image = cv2.imread(image_filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        a = random.choice(res['id'].values)
        label = res.loc[res['id'] == a, 'label'].values[0]

        if label > 0.5:
            label = 1
        else:
            label = 0

        ax.ravel()[i].imshow(image)
        ax.ravel()[i].set_title(class_[label])
        ax.ravel()[i].set_axis_off()

    plt.tight_layout()
    plt.show()

In [ ]:
# p.283 | 코드 6-38 | 테스트 이미지와 AlexNet의 예측 결과를 함께 출력

display_image_grid(test_images_filepaths)

In [ ]:
# p.286 | 코드 6-39 | VGGNet 구현에 필요한 라이브러리 호출

import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as Datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# p.286~287 | 변수에 객체를 그대로 대입했을 때 원본과 복사본의 값이 함께 변경되는지 확인

original = [1, 2, 3]
copy_o = original
print(copy_o)

copy_o[2] = 10
print(copy_o)
print(original)

In [ ]:
# p.287 | copy.copy()를 이용한 얕은 복사의 동작 확인

import copy

original = [[1, 2], 3]
copy_o = copy.copy(original)
print(copy_o)

copy_o[0] = 100
print(copy_o)
print(original)

append = copy.copy(original)
append[0].append(4)
print(append)
print(original)

In [13]:
# p.288 | copy.deepcopy()를 이용한 깊은 복사의 동작 확인

import copy

original = [[1, 2], 3]
copy_o = copy.deepcopy(original)
print(copy_o)

copy_o[0] = 100
print(copy_o)
print(original)

append = copy.deepcopy(original)
append[0].append(4)
print(append)
print(original)

[[1, 2], 3]
[100, 3]
[[1, 2], 3]
[[1, 2, 4], 3]
[[1, 2], 3]


In [14]:
# p.289 | 코드 6-40 | VGG 모델 클래스 정의

class VGG(nn.Module):
    def __init__(self, features, output_dim):
        super().__init__()
        self.features = features
        self.avgpool = nn.AdaptiveAvgPool2d(7)

        self.classifier = nn.Sequential(
            nn.Linear(512*7*7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, output_dim)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        h = x.view(x.shape[0], -1)
        x = self.classifier(h)

        return x, h

In [15]:
# p.289 | 코드 6-41 | VGG11, VGG13, VGG16, VGG19의 계층 구성을 리스트로 정의

vgg11_config = [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

vgg13_config = [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512,
                'M']

vgg16_config = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M',
                512, 512, 512, 'M']

vgg19_config = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512,
                512, 'M', 512, 512, 512, 512, 'M']

In [16]:
# p.290 | 코드 6-42 | config를 이용하여 VGG의 합성곱층과 풀링층을 생성하는 함수 정의

def get_vgg_layers(config, batch_norm):
    layers = []
    in_channels = 3

    for c in config:
        assert c == 'M' or isinstance(c, int)

        if c == 'M':
            layers += [nn.MaxPool2d(kernel_size = 2)]
        else:
            conv2d = nn.Conv2d(in_channels, c, kernel_size=3, padding=1)

            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(c), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]

            in_channels = c

    return nn.Sequential(*layers)

In [17]:
# p.291 | assert 뒤의 조건이 False일 때 오류가 발생하는지 확인

a = 1
assert a == 11

AssertionError: 

In [18]:
# p.291 | isinstance()를 이용하여 객체의 자료형을 확인

print(isinstance(1, int))
print(isinstance(1.2, int))
print(isinstance('deep learning', str))

True
False
True


In [19]:
# p.291 | 코드 6-43 | VGG11의 config를 이용하여 배치 정규화가 포함된 계층 생성

vgg11_layers = get_vgg_layers(vgg11_config, batch_norm=True)

In [20]:
# p.292 | 코드 6-44 | 생성한 VGG11 계층 구조 확인

print(vgg11_layers)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU(inplace=True)
  (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU(inplace=True)
  (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU(inplace=True)
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, cei

In [21]:
# p.293 | 코드 6-45 | VGG11 전체 네트워크 생성 및 구조 출력

OUTPUT_DIM = 2
model = VGG(vgg11_layers, OUTPUT_DIM)
print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): MaxPool2d(ke

In [22]:
# p.294 | 코드 6-46 | VGG11 사전 훈련된 모델 사용

import torchvision.models as models
pretrained_model = models.vgg11_bn(pretrained=True)
print(pretrained_model)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG11_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG11_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg11_bn-6002323d.pth" to /root/.cache/torch/hub/checkpoints/vgg11_bn-6002323d.pth


100%|██████████| 507M/507M [00:07<00:00, 67.8MB/s]


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): MaxPool2d(ke

In [23]:
# p.296 | 예제 코드 | VGG 모델 config 정의

vgg11_config = [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

vgg13_config = [64, 64, 'M', 128, 128, 'M', 256, 256, 'M',
                512, 512, 'M', 512, 512, 'M']

vgg16_config = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M',
                512, 512, 512, 'M', 512, 512, 512, 'M']

vgg19_config = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M',
                512, 512, 512, 512, 'M',
                512, 512, 512, 512, 'M']

My_Vgg = [64, 64, 64, 'M', 128, 128, 128, 'M', 256, 256, 256, 'M']

In [24]:
# p.296 | 코드 6-47 | 이미지 데이터 전처리

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [26]:
# p.297 | 예제 코드 | transforms.Compose 사용 예제

transforms.Compose([transforms.Resize((256, 256)),transforms.RandomRotation(5)])

Compose(
    Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)
    RandomRotation(degrees=[-5.0, 5.0], interpolation=nearest, expand=False, fill=0)
)

In [27]:
# p.297 | 코드 6-48 | ImageFolder를 이용하여 데이터셋 불러오기

train_path = '../chap06/data/catanddog/train'
test_path = '../chap06/data/catanddog/test'

train_dataset = torchvision.datasets.ImageFolder(
    train_path,
    transform=train_transforms
)

test_dataset = torchvision.datasets.ImageFolder(
    test_path,
    transform=test_transforms
)

print(len(train_dataset)), print(len(test_dataset))

FileNotFoundError: [Errno 2] No such file or directory: '../chap06/data/catanddog/train'

In [30]:
# p.298 | 코드 6-49 | 훈련과 검증 데이터 분할

VALID_RATIO = 0.9

n_train_examples = int(len(train_dataset) * VALID_RATIO)
n_valid_examples = len(train_dataset) - n_train_examples

train_data, valid_data = data.random_split(
    train_dataset,
    [n_train_examples, n_valid_examples]
)

NameError: name 'train_dataset' is not defined

In [28]:
# p.298 | 예제 코드 | random_split 사용 형식

train_data, valid_data = data.random_split(
    train_dataset,
    [n_train_examples, n_valid_examples]
)

NameError: name 'data' is not defined

In [29]:
# p.299 | 코드 6-50 | 검증 데이터 전처리

valid_data = copy.deepcopy(valid_data)
valid_data.dataset.transform = test_transforms

NameError: name 'valid_data' is not defined

In [ ]:
# p.299 | 코드 6-51 | 훈련, 검증, 테스트 데이터셋 수 확인

print(f'Number of training examples: {len(train_data)}')
print(f'Number of validation examples: {len(valid_data)}')
print(f'Number of testing examples: {len(test_dataset)}')

In [ ]:
# p.299 | 코드 6-52 | 메모리로 데이터 불러오기

BATCH_SIZE = 128

train_iterator = data.DataLoader(
    train_data,
    shuffle=True,
    batch_size=BATCH_SIZE
)

valid_iterator = data.DataLoader(
    valid_data,
    batch_size=BATCH_SIZE
)

test_iterator = data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

In [ ]:
# p.300 | 코드 6-53 | 옵티마이저와 손실 함수 정의

optimizer = optim.Adam(model.parameters(), lr=1e-7)
criterion = nn.CrossEntropyLoss()

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
# p.300 | 코드 6-54 | 모델 정확도 측정 함수

def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]

    return acc

In [ ]:
# p.300-301 | 코드 6-55 | 모델 학습 함수 정의

def train(model, iterator, optimizer, criterion, device):
    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for (x, y) in iterator:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred, _ = model(x)

        loss = criterion(y_pred, y)
        acc = calculate_accuracy(y_pred, y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
# p.301 | 코드 6-56 | 모델 성능 측정 함수

def evaluate(model, iterator, criterion, device):
    epoch_loss = 0
    epoch_acc = 0

    model.eval()

    with torch.no_grad():
        for (x, y) in iterator:
            x = x.to(device)
            y = y.to(device)

            y_pred, _ = model(x)

            loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
# p.301 | 코드 6-57 | 학습 시간 측정 함수

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))

    return elapsed_mins, elapsed_secs

In [ ]:
# p.302 | 코드 6-58 | 모델 학습

EPOCHS = 5

best_valid_loss = float('inf')

for epoch in range(EPOCHS):
    start_time = time.monotonic()

    train_loss, train_acc = train(
        model, train_iterator, optimizer, criterion, device
    )

    valid_loss, valid_acc = evaluate(
        model, valid_iterator, criterion, device
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), '../chap06/data/VGG-model.pt')

    end_time = time.monotonic()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Valid. Loss: {valid_loss:.3f} | Valid. Acc: {valid_acc*100:.2f}%')

In [31]:
# p.303 | 코드 6-59 | 테스트 데이터셋을 이용한 모델 성능 측정

model.load_state_dict(torch.load('../chap06/data/VGG-model.pt'))

test_loss, test_acc = evaluate(
    model, test_iterator, criterion, device
)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

FileNotFoundError: [Errno 2] No such file or directory: '../chap06/data/VGG-model.pt'

In [32]:
# p.303 | 코드 6-60 | 테스트 데이터셋을 이용한 모델의 예측 확인 함수

def get_predictions(model, iterator):
    model.eval()

    images = []
    labels = []
    probs = []

    with torch.no_grad():
        for (x, y) in iterator:
            x = x.to(device)

            y_pred, _ = model(x)
            y_prob = F.softmax(y_pred, dim=-1)

            top_pred = y_prob.argmax(1, keepdim=True)

            images.append(x.cpu())
            labels.append(y.cpu())
            probs.append(y_prob.cpu())

    images = torch.cat(images, dim=0)
    labels = torch.cat(labels, dim=0)
    probs = torch.cat(probs, dim=0)

    return images, labels, probs

In [33]:
# p.304 | 예제 코드 | torch.cat 사용 예제

import torch

x = torch.Tensor([[1, 2, 3], [2, 3, 4]])
y = torch.Tensor([[4, 5, 6], [5, 6, 7]])

print(torch.cat([x], dim=0))
print('---------------')

print(torch.cat([x, y]))
print('---------------')

print(torch.cat([x, y], dim=0))
print('---------------')

print(torch.cat([x, y], dim=1))

tensor([[1., 2., 3.],
        [2., 3., 4.]])
---------------
tensor([[1., 2., 3.],
        [2., 3., 4.],
        [4., 5., 6.],
        [5., 6., 7.]])
---------------
tensor([[1., 2., 3.],
        [2., 3., 4.],
        [4., 5., 6.],
        [5., 6., 7.]])
---------------
tensor([[1., 2., 3., 4., 5., 6.],
        [2., 3., 4., 5., 6., 7.]])


In [34]:
# p.305 | 코드 6-61 | 예측 중에서 정확하게 예측한 것을 추출

images, labels, probs = get_predictions(model, test_iterator)

pred_labels = torch.argmax(probs, 1)
corrects = torch.eq(labels, pred_labels)

correct_examples = []

for image, label, prob, correct in zip(
    images, labels, probs, corrects
):
    if correct:
        correct_examples.append((image, label, prob))

correct_examples.sort(
    reverse=True,
    key=lambda x: torch.max(x[2], dim=0).values
)

NameError: name 'test_iterator' is not defined

In [35]:
# p.305-306 | 예제 코드 | max와 argmax 반환값 확인

x = torch.rand(5, 3)
print(x)

print(torch.max(x))
print('---------------')
print(torch.argmax(x))

tensor([[0.4966, 0.4309, 0.1363],
        [0.7236, 0.4192, 0.9376],
        [0.9548, 0.2423, 0.9335],
        [0.5658, 0.3665, 0.0099],
        [0.3163, 0.8296, 0.2171]])
tensor(0.9548)
---------------
tensor(6)


In [36]:
# p.306 | 예제 코드 | zip 사용 예제

a = [1, 2, 3]
b = ['a', 'b', 'c']

for x, y in zip(a, b):
    print(x, y)

1 a
2 b
3 c


In [37]:
# p.307 | 예제 코드 | 리스트를 내림차순으로 정렬

x = [1, 7, 3, 9]

x.sort(reverse=True)

print('x:', x)

x: [9, 7, 3, 1]


In [38]:
# p.307 | 예제 코드 | lambda 함수

y = lambda x: x + 10

y(1)

11

In [39]:
# p.307 | 예제 코드 | torch.max에서 값과 인덱스 구하기

x = torch.randn([4, 4])
print(x)

max_elements, max_idxs = torch.max(x, dim=0)

print(max_elements)
print(max_idxs)

tensor([[ 0.3996, -0.9433,  0.0309,  0.2427],
        [-0.4609, -0.6305,  0.3589,  0.8299],
        [-0.3804, -0.7147,  2.3554, -1.2871],
        [ 0.1452, -1.2738, -0.7822, -1.3835]])
tensor([ 0.3996, -0.6305,  2.3554,  0.8299])
tensor([0, 1, 2, 1])


In [40]:
# p.309 | 코드 6-62 | 이미지 출력을 위한 전처리

def normalize_image(image):
    image_min = image.min()
    image_max = image.max()

    image.clamp_(min=image_min, max=image_max)
    image.add_(-image_min).div_(image_max-image_min+1e-5)

    return image

In [41]:
# p.309 | 예제 코드 | torch.add와 torch.add_ 차이 확인

x = torch.tensor([1, 2])

y = x.add(10)
print(y)
print(x is y)

print('-------------')

y = x.add_(10)
print(y)
print(x is y)

tensor([11, 12])
False
-------------
tensor([11, 12])
True


In [42]:
# p.310 | 코드 6-63 | 모델이 정확하게 예측한 이미지 출력 함수

def plot_most_correct(correct, classes, n_images, normalize=True):
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure(figsize=(25, 20))

    for i in range(rows * cols):
        ax = fig.add_subplot(rows, cols, i+1)

        image, true_label, probs = correct[i]

        image = image.permute(1, 2, 0)

        true_prob = probs[true_label]

        correct_prob, correct_label = torch.max(probs, dim=0)

        true_class = classes[true_label]
        correct_class = classes[correct_label]

        if normalize:
            image = normalize_image(image)

        ax.imshow(image.cpu().numpy())

        ax.set_title(
            f'true label: {true_class} ({true_prob:.3f})\n'
            f'pred label: {correct_class} ({correct_prob:.3f})'
        )

        ax.axis('off')

    fig.subplots_adjust(hspace=0.4)

In [43]:
# p.310 | 예제 코드 | permute로 텐서 차원 변경

x = torch.tensor([[1, 2, 3], [4, 5, 6]])

print(x)

x.permute(1, 0)

tensor([[1, 2, 3],
        [4, 5, 6]])


tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [44]:
# p.311 | 코드 6-64 | 예측결과 이미지 출력
classes = test_dataset.classes
N_IMAGES = 5
plot_most_correct(correct_examples, classes,N_IMAGES)

NameError: name 'tes_dataset' is not defined